# Zero-shot TimesFM on post-2018 SSMI test slice

Two baselines for direct comparison to the fine-tuned decomposed result: (1) raw zero-shot forecasting, (2) HP-filtered zero-shot forecasting (context-only decomposition, λ = 129600). Same sliding window (context = 120, horizon = 30, step = 30) as the fine-tuned notebooks. Both sections log to `TimesFM_PostTest_Baselines.log` and save separate `.npz` metric files.

## Section 1 — Raw zero-shot baseline

Pretrained TimesFM forecasts the raw 120-day context directly (no decomposition). Saves to `TimesFM_SSMI_PostTest_Baseline_Metrics.npz`.

In [1]:
import numpy as np
import pandas as pd
import timesfm
import logging
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from scipy.stats import pearsonr

logging.basicConfig(
    filename='TimesFM_PostTest_Baselines.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    force=True
)


def main():
    rmse_list = []
    mape_list = []
    pearson_list = []
    directional_hits = []
    try:
        # ========================
        # 1) Load SSMI test slice (post-2018)
        # ========================
        df = pd.read_csv("../DataSets/trimmed/SSMI.csv", parse_dates=["Date"])
        df = df.sort_values("Date").reset_index(drop=True)
        test_df = df[df["Date"] >= pd.Timestamp("2018-01-01")].reset_index(drop=True)
        test_df["Adj Close"] = test_df["Adj Close"].ffill()
        y = test_df["Adj Close"].values.astype(float)
        total_samples = len(y)
        logging.info(
            f"[RAW] Test range: {test_df['Date'].iloc[0].date()} -> {test_df['Date'].iloc[-1].date()} ({total_samples} days)"
        )
        print(
            f"[RAW] Test range: {test_df['Date'].iloc[0].date()} -> {test_df['Date'].iloc[-1].date()} ({total_samples} days)"
        )

        # ========================
        # 2) Sliding-window config
        # ========================
        context_window = 120
        forecast_horizon = 30
        step_size = 30
        num_segments = (total_samples - context_window) // step_size
        logging.info(f"[RAW] Segments to evaluate: {num_segments}")

        # ========================
        # 3) Load zero-shot TimesFM (no fine-tune)
        # ========================
        tfm = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend="cpu",
                per_core_batch_size=32,
                horizon_len=forecast_horizon,
            ),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id="google/timesfm-1.0-200m-pytorch",
            ),
        )
        logging.info("[RAW] Google TimesFM (zero-shot) loaded.")

        # ========================
        # 4) Sliding-window loop — forecast raw context
        # ========================
        for segment in range(num_segments):
            start_context = segment * step_size
            end_context = start_context + context_window
            if end_context + forecast_horizon > total_samples:
                break

            y_context = y[start_context:end_context]
            y_true = y[end_context:end_context + forecast_horizon]

            point_forecast, _ = tfm.forecast([y_context], freq=[0])
            pred = point_forecast[0][:forecast_horizon]

            prev_anchor = np.concatenate([[y[end_context - 1]], y_true[:-1]])
            actual_direction = np.sign(y_true - prev_anchor)
            pred_direction = np.sign(pred - prev_anchor)
            hits = (actual_direction == pred_direction).astype(int)
            directional_hits.extend(hits.tolist())

            rmse = np.sqrt(mean_squared_error(y_true, pred))
            mape = mean_absolute_percentage_error(y_true, pred) * 100
            r2 = pearsonr(y_true, pred).statistic ** 2

            rmse_list.append(rmse)
            mape_list.append(mape)
            pearson_list.append(r2)

            segment_dir_acc = hits.mean() * 100
            logging.info(
                f"[RAW] Segment {segment+1}/{num_segments}: RMSE={rmse:.4f}, MAPE={mape:.4f}%, R²={r2:.4f}, DirAcc={segment_dir_acc:.1f}%"
            )
            print(
                f"[RAW] Segment {segment+1}/{num_segments} — RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | R²: {r2:.4f} | Dir Acc: {segment_dir_acc:.1f}%"
            )

        # ========================
        # 5) Save results
        # ========================
        np.savez_compressed(
            "TimesFM_SSMI_PostTest_Baseline_Metrics.npz",
            rmse=np.array(rmse_list),
            mape=np.array(mape_list),
            pearson_coefficients=np.array(pearson_list),
            directional_hits=np.array(directional_hits),
            num_segments=num_segments,
            forecast_horizon=forecast_horizon,
            context_window=context_window,
        )
        logging.info("[RAW] Results saved to TimesFM_SSMI_PostTest_Baseline_Metrics.npz")

        # ========================
        # 6) Summary
        # ========================
        total_days = len(directional_hits)
        total_hits = sum(directional_hits)
        dir_acc_pct = (total_hits / total_days) * 100 if total_days else 0.0

        print("\n--- Median Metrics for Zero-Shot TimesFM on SSMI (post-2018, RAW) ---")
        print(f"Median RMSE:          {np.median(rmse_list):.4f}")
        print(f"Median MAPE:          {np.median(mape_list):.4f}%")
        print(f"Median Pearson R²:    {np.median(pearson_list):.4f}")
        print(f"Directional Accuracy: {total_hits}/{total_days} days ({dir_acc_pct:.2f}%)")

    except Exception:
        logging.error("[RAW] An error occurred.", exc_info=True)
        print("An error occurred. Check TimesFM_PostTest_Baselines.log for details.")
        try:
            np.savez_compressed(
                "partial_TimesFM_SSMI_PostTest_Baseline_Metrics.npz",
                rmse=np.array(rmse_list),
                mape=np.array(mape_list),
                pearson_coefficients=np.array(pearson_list),
                directional_hits=np.array(directional_hits),
            )
        except Exception:
            logging.error("[RAW] Failed to save partial results.", exc_info=True)
    finally:
        logging.info("[RAW] Forecasting run completed.")


if __name__ == '__main__':
    main()


 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
Loaded PyTorch TimesFM, likely because python version is 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)].
[RAW] Test range: 2018-01-03 -> 2021-05-17 (843 days)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[RAW] Segment 1/24 — RMSE: 324.37 | MAPE: 3.23% | R²: 0.9316 | Dir Acc: 43.3%
[RAW] Segment 2/24 — RMSE: 280.75 | MAPE: 2.84% | R²: 0.3634 | Dir Acc: 46.7%
[RAW] Segment 3/24 — RMSE: 148.85 | MAPE: 1.42% | R²: 0.1722 | Dir Acc: 56.7%
[RAW] Segment 4/24 — RMSE: 128.26 | MAPE: 1.10% | R²: 0.2023 | Dir Acc: 56.7%
[RAW] Segment 5/24 — RMSE: 277.03 | MAPE: 2.38% | R²: 0.0112 | Dir Acc: 66.7%
[RAW] Segment 6/24 — RMSE: 264.85 | MAPE: 2.51% | R²: 0.6728 | Dir Acc: 43.3%
[RAW] Segment 7/24 — RMSE: 88.95 | MAPE: 0.77% | R²: 0.6842 | Dir Acc: 50.0%
[RAW] Segment 8/24 — RMSE: 200.94 | MAPE: 1.82% | R²: 0.3346 | Dir Acc: 60.0%
[RAW] Segment 9/24 — RMSE: 78.27 | MAPE: 0.61% | R²: 0.0084 | Dir Acc: 63.3%
[RAW] Segment 10/24 — RMSE: 135.08 | MAPE: 1.09% | R²: 0.0049 | Dir Acc: 50.0%
[RAW] Segment 11/24 — RMSE: 164.91 | MAPE: 1.33% | R²: 0.1175 | Dir Acc: 53.3%
[RAW] Segment 12/24 — RMSE: 262.97 | MAPE: 2.35% | R²: 0.7147 | Dir Acc: 40.0%
[RAW] Segment 13/24 — RMSE: 131.63 | MAPE: 1.07% | R²: 0.7595 |

## Section 2 — HP-filter zero-shot baseline

Per segment: HP-decompose the context only (leakage-safe), forecast trend and cycle separately, recombine. Saves to `TimesFM_SSMI_PostTest_HP_Metrics.npz`.

In [2]:
import numpy as np
import pandas as pd
import timesfm
import logging
from statsmodels.tsa.filters.hp_filter import hpfilter
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from scipy.stats import pearsonr

logging.basicConfig(
    filename='TimesFM_PostTest_Baselines.log',
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    force=True
)


def hp_decompose_context(y_context, lamb):
    """HP decomposition on the 120-day context only — no look-ahead."""
    cycle, trend = hpfilter(y_context, lamb=lamb)
    return trend, cycle


def main():
    rmse_list = []
    mape_list = []
    pearson_list = []
    directional_hits = []
    try:
        # ========================
        # 1) Load SSMI test slice (post-2018)
        # ========================
        df = pd.read_csv("../DataSets/trimmed/SSMI.csv", parse_dates=["Date"])
        df = df.sort_values("Date").reset_index(drop=True)
        test_df = df[df["Date"] >= pd.Timestamp("2018-01-01")].reset_index(drop=True)
        test_df["Adj Close"] = test_df["Adj Close"].ffill()
        y = test_df["Adj Close"].values.astype(float)
        total_samples = len(y)
        logging.info(
            f"[HP] Test range: {test_df['Date'].iloc[0].date()} -> {test_df['Date'].iloc[-1].date()} ({total_samples} days)"
        )
        print(
            f"[HP] Test range: {test_df['Date'].iloc[0].date()} -> {test_df['Date'].iloc[-1].date()} ({total_samples} days)"
        )

        # ========================
        # 2) Sliding-window + HP config
        # ========================
        context_window = 120
        forecast_horizon = 30
        step_size = 30
        lamb = 129600
        num_segments = (total_samples - context_window) // step_size
        logging.info(f"[HP] Segments to evaluate: {num_segments} (lambda={lamb}, context-only)")

        # ========================
        # 3) Load zero-shot TimesFM (no fine-tune)
        # ========================
        tfm = timesfm.TimesFm(
            hparams=timesfm.TimesFmHparams(
                backend="cpu",
                per_core_batch_size=32,
                horizon_len=forecast_horizon,
            ),
            checkpoint=timesfm.TimesFmCheckpoint(
                huggingface_repo_id="google/timesfm-1.0-200m-pytorch",
            ),
        )
        logging.info("[HP] Google TimesFM (zero-shot) loaded.")

        # ========================
        # 4) Sliding-window loop — HP decompose context, forecast each component, recombine
        # ========================
        for segment in range(num_segments):
            start_context = segment * step_size
            end_context = start_context + context_window
            if end_context + forecast_horizon > total_samples:
                break

            y_context = y[start_context:end_context]
            y_true = y[end_context:end_context + forecast_horizon]

            context_low, context_high = hp_decompose_context(y_context, lamb=lamb)

            point_forecast_low, _ = tfm.forecast([context_low], freq=[0])
            median_low = point_forecast_low[0][:forecast_horizon]

            point_forecast_high, _ = tfm.forecast([context_high], freq=[0])
            median_high = point_forecast_high[0][:forecast_horizon]

            combined_pred = median_low + median_high

            prev_anchor = np.concatenate([[y[end_context - 1]], y_true[:-1]])
            actual_direction = np.sign(y_true - prev_anchor)
            pred_direction = np.sign(combined_pred - prev_anchor)
            hits = (actual_direction == pred_direction).astype(int)
            directional_hits.extend(hits.tolist())

            rmse = np.sqrt(mean_squared_error(y_true, combined_pred))
            mape = mean_absolute_percentage_error(y_true, combined_pred) * 100
            r2 = pearsonr(y_true, combined_pred).statistic ** 2

            rmse_list.append(rmse)
            mape_list.append(mape)
            pearson_list.append(r2)

            segment_dir_acc = hits.mean() * 100
            logging.info(
                f"[HP] Segment {segment+1}/{num_segments}: RMSE={rmse:.4f}, MAPE={mape:.4f}%, R²={r2:.4f}, DirAcc={segment_dir_acc:.1f}%"
            )
            print(
                f"[HP] Segment {segment+1}/{num_segments} — RMSE: {rmse:.2f} | MAPE: {mape:.2f}% | R²: {r2:.4f} | Dir Acc: {segment_dir_acc:.1f}%"
            )

        # ========================
        # 5) Save results
        # ========================
        np.savez_compressed(
            "TimesFM_SSMI_PostTest_HP_Metrics.npz",
            rmse=np.array(rmse_list),
            mape=np.array(mape_list),
            pearson_coefficients=np.array(pearson_list),
            directional_hits=np.array(directional_hits),
            num_segments=num_segments,
            forecast_horizon=forecast_horizon,
            context_window=context_window,
        )
        logging.info("[HP] Results saved to TimesFM_SSMI_PostTest_HP_Metrics.npz")

        # ========================
        # 6) Summary
        # ========================
        total_days = len(directional_hits)
        total_hits = sum(directional_hits)
        dir_acc_pct = (total_hits / total_days) * 100 if total_days else 0.0

        print("\n--- Median Metrics for Zero-Shot TimesFM on SSMI (post-2018, HP Filter) ---")
        print(f"Median RMSE:          {np.median(rmse_list):.4f}")
        print(f"Median MAPE:          {np.median(mape_list):.4f}%")
        print(f"Median Pearson R²:    {np.median(pearson_list):.4f}")
        print(f"Directional Accuracy: {total_hits}/{total_days} days ({dir_acc_pct:.2f}%)")

    except Exception:
        logging.error("[HP] An error occurred.", exc_info=True)
        print("An error occurred. Check TimesFM_PostTest_Baselines.log for details.")
        try:
            np.savez_compressed(
                "partial_TimesFM_SSMI_PostTest_HP_Metrics.npz",
                rmse=np.array(rmse_list),
                mape=np.array(mape_list),
                pearson_coefficients=np.array(pearson_list),
                directional_hits=np.array(directional_hits),
            )
        except Exception:
            logging.error("[HP] Failed to save partial results.", exc_info=True)
    finally:
        logging.info("[HP] Forecasting run completed.")


if __name__ == '__main__':
    main()


[HP] Test range: 2018-01-03 -> 2021-05-17 (843 days)


Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

[HP] Segment 1/24 — RMSE: 336.51 | MAPE: 3.34% | R²: 0.9252 | Dir Acc: 40.0%
[HP] Segment 2/24 — RMSE: 281.81 | MAPE: 2.93% | R²: 0.3663 | Dir Acc: 46.7%
[HP] Segment 3/24 — RMSE: 308.44 | MAPE: 2.83% | R²: 0.5229 | Dir Acc: 50.0%
[HP] Segment 4/24 — RMSE: 120.80 | MAPE: 1.05% | R²: 0.2165 | Dir Acc: 63.3%
[HP] Segment 5/24 — RMSE: 254.98 | MAPE: 2.38% | R²: 0.4461 | Dir Acc: 53.3%
[HP] Segment 6/24 — RMSE: 394.26 | MAPE: 3.85% | R²: 0.4118 | Dir Acc: 36.7%
[HP] Segment 7/24 — RMSE: 90.14 | MAPE: 0.68% | R²: 0.4119 | Dir Acc: 53.3%
[HP] Segment 8/24 — RMSE: 212.76 | MAPE: 1.94% | R²: 0.2747 | Dir Acc: 60.0%
[HP] Segment 9/24 — RMSE: 119.41 | MAPE: 1.00% | R²: 0.0224 | Dir Acc: 56.7%
[HP] Segment 10/24 — RMSE: 320.49 | MAPE: 3.01% | R²: 0.0001 | Dir Acc: 53.3%
[HP] Segment 11/24 — RMSE: 120.04 | MAPE: 1.05% | R²: 0.2377 | Dir Acc: 70.0%
[HP] Segment 12/24 — RMSE: 333.70 | MAPE: 2.99% | R²: 0.4719 | Dir Acc: 36.7%
[HP] Segment 13/24 — RMSE: 148.13 | MAPE: 1.26% | R²: 0.8007 | Dir Acc: 56